In [1]:
from rlm_sec.trainer import hf_dataloader


combined_qa = hf_dataloader.load_combined_qa()

/home/recoverx/astarag/recursive-lm-sec-filings/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
all_tickers_years = [(i,j) for i,j in zip(list(combined_qa['ticker_or_company_name']),list(combined_qa['year']))]

In [3]:
all_tickers_years = list(set(all_tickers_years))

In [9]:
# from concurrent.futures import ThreadPoolExecutor, as_completed
# from rlm_sec.filings import sec_data
# from rlm_sec.filings.utils import company_to_ticker
# import asyncio


# # Function to call sec_main for a given (ticker, year)
# def fetch_sec_main(args):
#     ticker, year = args
#     ticker = company_to_ticker(ticker)
#     # sec_main is async, so run it with asyncio
#     if not ticker:
#         return None, None, None
#     return (ticker, year, asyncio.run(sec_data.sec_main(ticker, year)))

# results = []
# with ThreadPoolExecutor() as executor:
#     # Submit all (ticker, year) pairs for execution
#     futures = [executor.submit(fetch_sec_main, (ticker, year)) for ticker, year in all_tickers_years]
#     for future in as_completed(futures):
#         try:
#             ticker, year, value = future.result()
#             if not ticker:
#                 continue
#             results.append((ticker, year, value))
#         except Exception as e:
#             print(f"Error fetching ({ticker}, {year}): {e}")


In [1]:
import re
from pathlib import Path
from settings import olmocr_settings
from rlm_sec.dataloader.vector_store import FaissVectorIndex

# Root containing one directory per "TICKER-YYYY" with *.md inside each.
MARKDOWN_ROOT = Path("localworkspace/markdown")
FORCE_REBUILD = False

# Directory names must end with -YYYY (handles tickers like BRK-B-2025).
_TICKER_YEAR_DIR = re.compile(r"^(?P<ticker>.+)-(?P<year>\d{4})$")

index = FaissVectorIndex()
all_keys = []

for sub in sorted(MARKDOWN_ROOT.iterdir()):
    if not sub.is_dir():
        continue
    m = _TICKER_YEAR_DIR.match(sub.name)
    if not m:
        print(f"skip (not TICKER-YYYY): {sub.name}")
        continue
    ticker, year = m["ticker"], m["year"]
    md_paths = sorted(sub.glob("*.md"))
    if not md_paths:
        print(f"skip (no .md): {sub.name}")
        continue
    try:
        keys = index.from_markdown(
            ticker=ticker,
            year=year,
            markdown_paths=md_paths,
            force=FORCE_REBUILD,
        )
    except Exception:
        pass
    all_keys.extend(keys)
    print(f"indexed {sub.name}: {len(keys)} filing(s)")

print(f"total index keys: {len(all_keys)}")

/home/recoverx/astarag/recursive-lm-sec-filings/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/recoverx/astarag/recursive-lm-sec-filings/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


ModuleNotFoundError: No module named 'faiss'